<a href="https://colab.research.google.com/github/Sarkis55/Air_Quality_Predictor/blob/main/BSAN6070_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Combine the 2021, 2022, 2023, 2024, and 2025 PM2.5 California Datasets from https://www.epa.gov/outdoor-air-quality-data/download-daily-data

import pandas as pd
import glob

# Find all CSV files that match the naming pattern
files = glob.glob("ad_viz_plotval_data*.csv")

print("Files found:", files)

# Combine them
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Shape:", df.shape)
df.head()

Files found: ['ad_viz_plotval_data_2022.csv', 'ad_viz_plotval_data_2023.csv', 'ad_viz_plotval_data_2024.csv', 'ad_viz_plotval_data_2021.csv', 'ad_viz_plotval_data_2025.csv']
Shape: (295383, 22)


,Date,Source,Site ID,POC,Daily Mean PM2.5 Concentration,Units,Daily AQI Value,Local Site Name,Daily Obs Count,Percent Complete,...,Method Code,Method Description,CBSA Code,CBSA Name,State FIPS Code,State,County FIPS Code,County,Site Latitude,Site Longitude
0,01/01/2022,AQS,60010007,3,12.7,ug/m3 LC,58,Livermore,1,100.0,...,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41860.0,"San Francisco-Oakland-Hayward, CA",6,California,1,Alameda,37.687526,-121.784217
1,01/02/2022,AQS,60010007,3,13.9,ug/m3 LC,60,Livermore,1,100.0,...,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41860.0,"San Francisco-Oakland-Hayward, CA",6,California,1,Alameda,37.687526,-121.784217
2,01/03/2022,AQS,60010007,3,7.1,ug/m3 LC,39,Livermore,1,100.0,...,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41860.0,"San Francisco-Oakland-Hayward, CA",6,California,1,Alameda,37.687526,-121.784217
3,01/04/2022,AQS,60010007,3,3.7,ug/m3 LC,21,Livermore,1,100.0,...,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41860.0,"San Francisco-Oakland-Hayward, CA",6,California,1,Alameda,37.687526,-121.784217
4,01/05/2022,AQS,60010007,3,4.2,ug/m3 LC,23,Livermore,1,100.0,...,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41860.0,"San Francisco-Oakland-Hayward, CA",6,California,1,Alameda,37.687526,-121.784217


In [ ]:
df.columns

Index(['Date', 'Source', 'Site ID', 'POC', 'Daily Mean PM2.5 Concentration',
       'Units', 'Daily AQI Value', 'Local Site Name', 'Daily Obs Count',
       'Percent Complete', 'AQS Parameter Code', 'AQS Parameter Description',
       'Method Code', 'Method Description', 'CBSA Code', 'CBSA Name',
       'State FIPS Code', 'State', 'County FIPS Code', 'County',
       'Site Latitude', 'Site Longitude'],
      dtype='object')

In [ ]:
#only keep columns that are necessary

df = df[[
    'Date',
    'Daily AQI Value',
    'Daily Mean PM2.5 Concentration',
    'State',
    'County',
    'Site Latitude',
    'Site Longitude'
]]

In [ ]:
#rename some columns
df = df.rename(columns={
    'Daily AQI Value': 'AQI',
    'Daily Mean PM2.5 Concentration': 'PM25'
})

In [ ]:
# Convert date
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y', errors='coerce')

# Drop rows where date failed OR AQI missing
df = df.dropna(subset=['Date', 'AQI', 'PM25'])

# Remove duplicates
df = df.drop_duplicates()

In [ ]:
df.describe()

,Date,AQI,PM25,Site Latitude,Site Longitude,AQI_Category,Year,Month,Season
count,290533,290533.000000,290533.000000,290532.000000,290532.000000,290533.000000,290533.000000,290533.000000,290533.000000
mean,2023-06-20 15:05:30.654349056,39.124309,8.439220,36.286682,-119.653954,0.010587,2022.968971,6.512407,2.502745
min,2021-01-01 00:00:00,0.000000,-6.700000,32.552824,-124.203470,0.000000,2021.000000,1.000000,1.000000
25%,2022-03-21 00:00:00,23.000000,4.100000,34.069570,-121.493110,0.000000,2022.000000,4.000000,2.000000
50%,2023-06-09 00:00:00,37.000000,6.700000,36.489470,-119.706200,0.000000,2023.000000,7.000000,3.000000
75%,2024-09-19 00:00:00,54.000000,10.500000,37.962069,-117.938450,0.000000,2024.000000,10.000000,3.000000
max,2025-12-31 00:00:00,1435.000000,794.900000,41.756130,-115.483070,1.000000,2025.000000,12.000000,4.000000
std,NaN,23.044090,9.061169,2.324674,2.040379,0.102349,1.413969,3.444965,1.114219


In [ ]:
# Remove invalid values
df = df[(df['PM25'] >= 0) & (df['AQI'] >= 0)]

#Remove extreme outliers
df = df[df['PM25'] < df['PM25'].quantile(0.99)]

In [ ]:
df.head()
df['Date'].head()

,Date
0,2024-01-01
1,2024-01-02
2,2024-01-03
3,2024-01-04
4,2024-01-05


In [ ]:
# Create Target Variable
# 1 = Unhealthy, 0 = Safe

df['AQI_Category'] = df['AQI'].apply(lambda x: 1 if x > 75 else 0)

df['AQI_Category'].value_counts()

,count
AQI_Category,
0,278411
1,1983


In [ ]:
# Feature Engineering

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Season (VERY important for pollution patterns)
df['Season'] = df['Month'] % 12 // 3 + 1

In [ ]:
# Add location feature
df['County_Code'] = df['County'].astype('category').cat.codes

# Scale PM2.5
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['PM25'] = scaler.fit_transform(df[['PM25']])

In [ ]:
# Define Features + Target

features = ['PM25', 'Month', 'Season', 'County_Code']
X = df[features]
y = df['AQI_Category']